# 07 — Alert outcomes and realised lead time

Fills `gold_alert_outcome` by checking what happened **after** each alert.

**Definitions (first MVP, H = 7 days)**

| Outcome | Rule |
|---|---|
| **Confirmed — coded onset** | A coded deferment episode starts **after** the alert date and within 7 days, at the same station. |
| **Confirmed — large-loss onset** | Deferred oil first reaches or exceeds the station’s own p90 **after** the alert, within 7 days, after 3 quiet observed days below p90. |
| **False alarm** | The full 7-day window is available and neither event occurs. |
| **Pending** | The 7-day window extends beyond the data and no event has occurred yet. |

`actual_lead_days = event_date − alert_date` (1–7 for confirmed futures).

**Safety**

- Existing manual fields are preserved: `acknowledged_by`, `acknowledged_at_utc`,
  `action_note`, `false_reason`, and any non-computed `disposition`.
- The notebook uses a Delta `MERGE` by `alert_id`, so reruns update the
  computed outcome without deleting manual feedback.
- A `pending` row can later become `confirmed` or `false_alarm`.


In [ ]:
from datetime import datetime, date, timezone
import pandas as pd
from pyspark.sql.types import (
    BooleanType, DateType, LongType, StringType, StructField, StructType,
    TimestampType,
)

WORKSPACE_ID = "355cd573-8b17-430a-b606-c6b9424d36a1"
SILVER_ID = "b6d8fe8b-bad2-4ac5-a605-f0b13893df7f"
GOLD_ID = "df9a08e5-b370-4c2a-b608-6d0f326003ef"
SILVER = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"
GOLD = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{GOLD_ID}"

HORIZON_DAYS = 7
QUIET_DAYS = 3
COMPUTED_DISPOSITIONS = {"pending"}


def read_gold(name: str):
    return spark.read.format("delta").load(f"{GOLD}/Tables/{name}")


def read_silver(name: str):
    return spark.read.format("delta").load(f"{SILVER}/Tables/{name}")


alerts = read_gold("gold_alert").select(
    "alert_id", "station_id", "station_name", "alert_date", "tier"
).toPandas()

episodes = read_silver("silver_deferment_episode").filter(
    "has_coded_event = true"
).select("station_id", "episode_start", "cause_code", "cause_group").toPandas()

risk = read_gold("gold_risk_daily").filter(
    "is_observed = true"
).select("station_id", "date", "deferred_oil").toPandas()

baseline = read_gold("gold_entity_baseline").filter(
    "metric = 'deferred_oil'"
).select("entity_id", "p90").toPandas()

alerts["alert_date"] = pd.to_datetime(alerts["alert_date"])
episodes["episode_start"] = pd.to_datetime(episodes["episode_start"])
risk["date"] = pd.to_datetime(risk["date"])
p90 = dict(zip(baseline["entity_id"], baseline["p90"]))

max_data_date = risk["date"].max()
print(f"alerts: {len(alerts)}")
print(f"coded episodes: {len(episodes)}")
print(f"observed risk rows: {len(risk)}")
print(f"data ends: {max_data_date.date()}")


In [ ]:
# --- 1. large-loss onsets: p90 exceeded after 3 quiet observed days --------
risk = risk.sort_values(["station_id", "date"]).reset_index(drop=True)
loss_rows = []
for station_id, group in risk.groupby("station_id"):
    group = group.sort_values("date").reset_index(drop=True)
    threshold = p90.get(station_id)
    if threshold is None or pd.isna(threshold):
        continue
    for _, row in group.iterrows():
        if pd.isna(row["deferred_oil"]) or row["deferred_oil"] < threshold:
            continue
        previous = group[group["date"] < row["date"]].tail(QUIET_DAYS)
        if len(previous) == QUIET_DAYS and (previous["deferred_oil"] < threshold).all():
            loss_rows.append({"station_id": station_id, "date": row["date"]})
large_loss = pd.DataFrame(loss_rows)
print(f"large-loss onsets: {len(large_loss)}")

# --- 2. outcome per alert ---------------------------------------------------
outcome_rows = []
now = datetime.now(timezone.utc).replace(tzinfo=None)

for _, alert in alerts.iterrows():
    start = alert["alert_date"].normalize()
    end = start + pd.Timedelta(days=HORIZON_DAYS)

    coded = episodes[
        (episodes["station_id"] == alert["station_id"])
        & (episodes["episode_start"] > start)
        & (episodes["episode_start"] <= end)
    ]
    first_coded = coded["episode_start"].min() if len(coded) else pd.NaT

    loss = large_loss[
        (large_loss["station_id"] == alert["station_id"])
        & (large_loss["date"] > start)
        & (large_loss["date"] <= end)
    ] if len(large_loss) else large_loss
    first_loss = loss["date"].min() if len(loss) else pd.NaT

    candidates = []
    if pd.notna(first_coded):
        candidates.append((first_coded, "coded_onset"))
    if pd.notna(first_loss):
        candidates.append((first_loss, "large_loss"))

    if candidates:
        event_date, detection = sorted(
            candidates,
            key=lambda x: (x[0], 0 if x[1] == "coded_onset" else 1),
        )[0]
        lead_days = (event_date - start).days
        event_occurred = True
        disposition = "confirmed"
    else:
        event_date = None
        lead_days = None
        if end > max_data_date:
            event_occurred = None
            detection = "pending_horizon"
            disposition = "pending"
        else:
            event_occurred = False
            detection = "none"
            disposition = "false_alarm"

    outcome_rows.append((
        alert["alert_id"],
        now,
        event_occurred,
        event_date.date() if event_date is not None else None,
        lead_days,
        detection,
        disposition,
        None,
        None,
        None,
        None,
        now,
    ))

schema = StructType([
    StructField("alert_id", StringType(), False),
    StructField("outcome_recorded_at_utc", TimestampType(), False),
    StructField("event_occurred", BooleanType(), True),
    StructField("event_date", DateType(), True),
    StructField("actual_lead_days", LongType(), True),
    StructField("detection_type", StringType(), False),
    StructField("disposition", StringType(), False),
    StructField("acknowledged_by", StringType(), True),
    StructField("acknowledged_at_utc", TimestampType(), True),
    StructField("action_note", StringType(), True),
    StructField("false_reason", StringType(), True),
    StructField("updated_at_utc", TimestampType(), False),
])
outcomes = spark.createDataFrame(outcome_rows, schema)
outcomes.createOrReplaceTempView("df_alert_outcomes_new")
print(f"computed outcomes: {outcomes.count()}")
display(outcomes.toPandas())


In [ ]:
# --- 3. merge into gold_alert_outcome, preserving manual feedback ----------
path = f"{GOLD}/Tables/gold_alert_outcome"
if not spark.catalog.tableExists("gold_alert_outcome"):
    (outcomes.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(path))
    spark.sql(f"CREATE TABLE IF NOT EXISTS gold_alert_outcome USING DELTA LOCATION '{path}'")
    print("created gold_alert_outcome")
else:
    spark.sql('''
        MERGE INTO gold_alert_outcome AS t
        USING df_alert_outcomes_new AS s
        ON t.alert_id = s.alert_id
        WHEN MATCHED THEN UPDATE SET
            t.outcome_recorded_at_utc = s.outcome_recorded_at_utc,
            t.event_occurred = s.event_occurred,
            t.event_date = s.event_date,
            t.actual_lead_days = s.actual_lead_days,
            t.detection_type = s.detection_type,
            t.disposition = CASE
                WHEN t.disposition IS NULL OR t.disposition = '' OR t.disposition = 'pending'
                THEN s.disposition ELSE t.disposition END,
            t.updated_at_utc = s.updated_at_utc
        WHEN NOT MATCHED THEN INSERT (
            alert_id, outcome_recorded_at_utc, event_occurred, event_date,
            actual_lead_days, detection_type, disposition, acknowledged_by,
            acknowledged_at_utc, action_note, false_reason, updated_at_utc
        ) VALUES (
            s.alert_id, s.outcome_recorded_at_utc, s.event_occurred, s.event_date,
            s.actual_lead_days, s.detection_type, s.disposition, s.acknowledged_by,
            s.acknowledged_at_utc, s.action_note, s.false_reason, s.updated_at_utc
        )
    ''')
    print("merged gold_alert_outcome")

out = spark.table("gold_alert_outcome").toPandas()
out["event_occurred"] = out["event_occurred"].astype("boolean")
evaluated = out[out["event_occurred"].notna()]
confirmed = out[out["event_occurred"] == True]  # noqa: E712
false_alarms = out[out["event_occurred"] == False]  # noqa: E712
pending = out[out["disposition"] == "pending"]

print(f"outcome rows      : {len(out)}")
print(f"evaluated         : {len(evaluated)}")
print(f"confirmed         : {len(confirmed)}")
print(f"false alarms      : {len(false_alarms)}")
print(f"pending           : {len(pending)}")
if len(confirmed):
    print(f"median lead (days): {confirmed['actual_lead_days'].median():.1f}")
    print(f"mean lead (days)  : {confirmed['actual_lead_days'].mean():.1f}")
if len(evaluated):
    print(f"precision         : {len(confirmed) / len(evaluated):.1%}")
    print(f"false-alarm rate  : {len(false_alarms) / len(evaluated):.1%}")

display(out.sort_values("outcome_recorded_at_utc", ascending=False))
